# Match observations and temporal features
Run this notebook from the notebooks directory with the project installed. All teams and outcomes here are fictional. Outputs are generated by your run, not prefilled evidence.
Learning goals: source revision vs arrival order, idempotent ingestion, quarantine, temporal features, and honest baseline comparison.

In [ ]:
from pathlib import Path
from arena_lakehouse.synthetic import generate
from arena_lakehouse.engine import Pipeline, latest, team_stats, history
from arena_lakehouse.prediction import features, train_and_predict
from arena_lakehouse.snapshot import export_snapshot
workspace = Path('../data/notebook-01')
pipeline = Pipeline(workspace)
events = generate(matches=240, seed=42)
pipeline.ingest(events)

## 1. Revisions and completed-match statistics
Input is intentionally shuffled. Current state uses the highest accepted revision, not the last row in the input. A completed football draw and a VALORANT map score have different meanings.

In [ ]:
canonical = pipeline.events()
matches = latest(canonical)
team_stats(matches)[:4]

In [ ]:
before = pipeline.events()
retry = pipeline.ingest(events)
assert pipeline.events() == before
assert retry['accepted'] == 0
pipeline.quality()

## 2. What was known at kickoff?
Only results completed before the target kickoff AND observed before the cutoff may enter a feature. Training labels must also be available at fit time. Fetching historical data today cannot manufacture old observation timestamps.

In [ ]:
target = sorted([m for m in matches if m['status'] == 'finished'], key=lambda m: m['start_time'])[80]
feature_values, evidence = features(target, canonical, target['start_time'])
feature_values, evidence

## 3. Evaluate a baseline without test tuning
Compare accuracy, log loss and Brier score against the training-frequency baseline. A higher accuracy can coexist with worse probability quality. Synthetic scores measure the workflow, not real-team predictive skill.

In [ ]:
predictions, report = train_and_predict(canonical)
report

In [ ]:
snapshot = export_snapshot(pipeline, workspace / 'snapshots')
print(snapshot.resolve())
predictions[:1]

## Next engineering challenge
Introduce a cancellation and a late score correction. Identify every affected match/team key. In the next milestone, update only those Silver/Gold rows and compare against a full reconstruction. Use the Docker smoke for actual Spark/Delta, checkpoint and CDF evidence.